# Fine-Tuning Qwen2 for Text Summarization using QLoRA

## Overview

This notebook demonstrates the complete process of fine-tuning a large language model (Qwen2-0.5B-Instruct) for text summarization using QLoRA (Quantized Low-Rank Adaptation). We'll cover the entire pipeline from data preparation to model evaluation, providing detailed explanations of each component and design decision.

### What is QLoRA?

QLoRA combines two key efficiency techniques:
- **Quantization**: Reduces model precision from 16-bit to 4-bit, dramatically reducing memory requirements
- **LoRA (Low-Rank Adaptation)**: Adds small trainable adapter layers instead of fine-tuning all parameters

This approach enables fine-tuning large models on consumer hardware while maintaining performance quality.

### Key Learning Objectives

By the end of this notebook, you'll understand:
1. How 4-bit quantization reduces memory footprint without significant quality loss
2. Why LoRA enables parameter-efficient fine-tuning
3. How to structure instruction-following datasets for decoder-only models
4. Best practices for hyperparameter selection in QLoRA training
5. Methods for evaluating fine-tuned summarization models

### Technical Stack

- **Model**: Qwen2-0.5B-Instruct (compact but capable instruction-tuned model)
- **Dataset**: CNN/DailyMail (standard benchmark for abstractive summarization)
- **Training Framework**: Hugging Face Transformers + TRL (Transformer Reinforcement Learning)
- **Efficiency Techniques**: 4-bit quantization + LoRA adapters


## Environment Setup

### Required Dependencies

The following packages are essential for QLoRA fine-tuning:

- **bitsandbytes**: Enables 4-bit quantization and 8-bit optimizers
- **trl**: Transformer Reinforcement Learning library for supervised fine-tuning
- **rouge_score**: Evaluation metrics for summarization tasks
- **evaluate**: Hugging Face evaluation framework
- **peft**: Parameter-Efficient Fine-Tuning library for LoRA implementation

In [2]:
!pip install -U bitsandbytes trl rouge_score evaluate

## Model Loading and Quantization

### Why Qwen2-0.5B-Instruct?

We selected Qwen2-0.5B-Instruct for several strategic reasons:

1. **Instruction-tuned**: Pre-trained to follow conversational instructions, making it ideal for summarization tasks
2. **Compact size**: 0.5B parameters allow experimentation on limited hardware while demonstrating concepts
3. **Strong baseline**: Despite its size, shows competitive performance on downstream tasks
4. **Efficient architecture**: Modern transformer design with optimizations for inference speed

### 4-bit Quantization Configuration

Quantization reduces memory usage by representing model weights with lower precision. Our configuration uses several advanced techniques:

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2-0.5B-Instruct"

# 4-bit quantization configuration - the "Q" in QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Enable 4-bit precision loading
    bnb_4bit_quant_type="nf4",           # NormalFloat4: theoretically optimal 4-bit data type
    bnb_4bit_compute_dtype="float16",     # Computation precision for activations
    bnb_4bit_use_double_quant=True,      # Double quantization: quantize quantization constants
)

# Memory Impact: Standard FP16 model ~1GB, 4-bit quantized ~250MB (75% reduction)

# Load the quantized model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",                    # Automatically distribute across available GPUs
    quantization_config=bnb_config,
)

# Training optimizations
model.config.use_cache = False           # Disable KV cache to save memory during training
model.gradient_checkpointing_enable()    # Trade compute for memory: recompute activations during backprop

# Load tokenizer and handle padding
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Use EOS token for padding when needed

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Data Preparation and Instruction Formatting

### Dataset Selection: CNN/DailyMail

We use the CNN/DailyMail dataset for several reasons:

1. **Quality**: Professional journalism provides high-quality article-summary pairs
2. **Scale**: Large dataset (300k+ examples) allows robust training and evaluation
3. **Benchmark**: Standard dataset for comparing summarization model performance
4. **Diversity**: Covers various news topics, improving model generalization

### Instruction Format for Decoder-Only Models

Unlike encoder-decoder models (T5, BART), decoder-only models like Qwen2 require special formatting to understand the task structure. We use a conversational format that clearly delineates input and expected output.

In [ ]:
from datasets import load_dataset

# Load and sample the dataset for efficient experimentation
dataset = (
    load_dataset("cnn_dailymail", '3.0.0', split="train")
      .shuffle(seed=42)                    # Ensure reproducible sampling
      .select(range(3_000))                # Use 3k examples for faster training
)

def create_instruction_prompt(example):
    """
    Convert article-summary pairs into instruction-following format.
    
    The ChatML format used here:
    - <|im_start|>user: Begins user instruction
    - <|im_end|>: Ends current speaker turn  
    - <|im_start|>assistant: Begins model response

    (im stands for "Instant Message")
    
    This format teaches the model when to respond and what constitutes a complete response.
    """
    prompt_template = (
        "<|im_start|>user\n"
        "Summarize the following article:\n\n{article}<|im_end|>\n"
        "<|im_start|>assistant\n{summary}<|im_end|>"
    )
    
    return {"text": prompt_template.format(
        article=example["article"], 
        summary=example["highlights"]
    )}

# Transform dataset to instruction format
formatted_dataset = dataset.map(
    create_instruction_prompt, 
    remove_columns=dataset.column_names  # Remove original columns, keep only 'text'
)

print(f"Example formatted prompt (first 500 chars):")
print(formatted_dataset[0]["text"][:500] + "...")

In [ ]:
# Create train/validation/test splits following ML best practices
total_size = len(formatted_dataset)
train_size = int(0.7 * total_size)      # 70% for training (2,100 examples)
eval_size = int(0.15 * total_size)      # 15% for validation during training (450 examples)
test_size = total_size - train_size - eval_size  # 15% for final evaluation (450 examples)

train_dataset = formatted_dataset.select(range(train_size))
eval_dataset = formatted_dataset.select(range(train_size, train_size + eval_size))
test_dataset = formatted_dataset.select(range(train_size + eval_size, total_size))

print(f"Dataset splits:")
print(f"  Training examples: {len(train_dataset)}")
print(f"  Validation examples: {len(eval_dataset)}")  
print(f"  Test examples: {len(test_dataset)}")

# The validation set guides training decisions (early stopping, hyperparameter tuning)
# The test set provides unbiased final performance evaluation

Training examples: 2100
Evaluation examples: 450
Test examples: 450


## LoRA Configuration: Parameter-Efficient Fine-Tuning

### Understanding LoRA (Low-Rank Adaptation)

LoRA works by freezing the original model weights and adding small trainable matrices to specific layers. Instead of updating all parameters, we train only these adapter matrices, dramatically reducing:

- **Memory requirements**: Only adapter weights need gradients
- **Storage costs**: Save only the small adapter weights (~10MB vs 1GB+ for full model)
- **Training time**: Fewer parameters to optimize

### Hyperparameter Selection Rationale

The LoRA configuration below balances expressiveness with efficiency:

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

peft_config = LoraConfig(
    r=32,                                 # Rank: Higher = more expressive but larger adapters
                                         # 32 is a good balance for most tasks (range: 8-128)
    
    lora_alpha=16,                       # Scaling factor: Controls adaptation strength
                                         # Common practice: alpha = r/2 for stable training
    
    lora_dropout=0.1,                    # Regularization: Prevents overfitting in adapters
                                         # 0.1 is standard for most tasks
    
    bias="none",                         # Don't adapt bias terms (saves parameters)
    task_type="CAUSAL_LM",              # Specify causal language modeling task
    
    # Target modules: Apply LoRA to all linear layers in attention and MLP blocks
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",    # Attention projections
        "gate_proj", "up_proj", "down_proj"        # MLP layers
    ]
)

# Prepare model for k-bit training (integrates with quantization)
model = prepare_model_for_kbit_training(model)

# Calculate trainable parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params:,}")
print(f"Estimated LoRA parameters: ~{(32 * 2 * len(peft_config.target_modules) * 896):,}")  # Rough estimate
print(f"Trainable ratio: ~{(32 * 2 * len(peft_config.target_modules) * 896) / total_params * 100:.2f}%")

## Training Configuration: Optimizing for Quality and Efficiency

### Training Strategy Design

Our training configuration balances several competing objectives:

1. **Memory efficiency**: Small batch sizes with gradient accumulation
2. **Training stability**: Learning rate scheduling and gradient clipping
3. **Quality monitoring**: Regular evaluation and early stopping
4. **Computational efficiency**: Mixed precision and gradient checkpointing

### Key Hyperparameter Decisions

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

training_arguments = SFTConfig(
    output_dir="output",
    
    # Batch size strategy: Small batches + accumulation = effective large batch
    per_device_train_batch_size=2,      # Fits in GPU memory with quantization
    per_device_eval_batch_size=2,       # Match training batch size
    gradient_accumulation_steps=4,      # Effective batch size = 2 * 4 = 8
    
    # Optimizer selection: Memory-efficient 8-bit AdamW
    optim="paged_adamw_8bit",          # Reduces optimizer memory by ~50%
    
    # Learning rate configuration
    learning_rate=2e-4,                 # Higher LR for LoRA (vs 5e-5 for full fine-tuning)
    lr_scheduler_type="linear",         # Linear decay from peak to 0
    warmup_ratio=0.1,                   # 10% warmup prevents early instability
    
    # Training duration
    num_train_epochs=3,                 # Usually sufficient for LoRA adaptation
    
    # Stability measures
    max_grad_norm=0.3,                  # Gradient clipping prevents exploding gradients
    fp16=True,                          # Mixed precision: 2x speedup, minimal quality loss
    gradient_checkpointing=True,        # Trade compute for memory during backprop
    
    # Dataset configuration
    dataset_text_field="text",          # Column name in our formatted dataset
    
    # Evaluation and monitoring
    eval_strategy="steps",              # Evaluate during training (not just at end)
    eval_steps=50,                      # Check progress every 50 steps
    logging_steps=10,                   # Log metrics frequently for monitoring
    
    # Model saving strategy
    save_strategy="steps",              # Save checkpoints during training
    save_steps=50,                      # Save every 50 steps (aligned with eval)
    save_total_limit=2,                 # Keep only best 2 checkpoints (save disk space)
    load_best_model_at_end=True,        # Load best checkpoint after training
    metric_for_best_model="eval_loss",  # Use validation loss for model selection
    greater_is_better=False,            # Lower loss = better model
    
    # Experiment tracking
    report_to="wandb",                  # Log to Weights & Biases for visualization
    run_name="qwen-qlora-summarization", # Descriptive run name
)

## Model Training and Monitoring

### Training Process Overview

The SFTTrainer orchestrates the entire training pipeline:

1. **Initialization**: Combines model, tokenizer, datasets, and configuration
2. **Training Loop**: Iterates through batches, computing loss and gradients
3. **Evaluation**: Periodically assesses performance on validation set
4. **Checkpointing**: Saves best models based on validation metrics
5. **Early Stopping**: Prevents overfitting by stopping when validation loss plateaus

In [ ]:
# Initialize the trainer with all components
trainer = SFTTrainer(
    model=model,                        # Quantized base model
    processing_class=tokenizer,         # Tokenizer for text processing
    train_dataset=train_dataset,        # Training data (2,100 examples)
    eval_dataset=eval_dataset,          # Validation data (450 examples)
    args=training_arguments,            # All training hyperparameters
    peft_config=peft_config,           # LoRA configuration
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,  # Stop if no improvement for 3 evaluations
        )
    ]
)

print("Starting training...")
print(f"Total training steps: {len(train_dataset) // (training_arguments.per_device_train_batch_size * training_arguments.gradient_accumulation_steps) * training_arguments.num_train_epochs}")

# Execute training loop
training_output = trainer.train()

# Display training summary
print(f"\nTraining completed!")
print(f"Final training loss: {training_output.training_loss:.4f}")
print(f"Training steps completed: {training_output.global_step}")

# Save the trained LoRA adapter (only ~10MB vs full model ~500MB)
trainer.model.save_pretrained("qwen-qlora-adapter")
tokenizer.save_pretrained("qwen-qlora-adapter")
print("Model adapter saved to: qwen-qlora-adapter/")

wandb: Currently logged in as: maya-sam (maya-sam-mayaai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,2.657900,2.643092,2.654487,325968.000000,0.457632
100,2.651100,2.616083,2.640901,652592.000000,0.461338
150,2.601700,2.604198,2.628846,979976.000000,0.462867
200,2.576600,2.596648,2.597197,1307897.000000,0.463968
250,2.564300,2.590667,2.572707,1630660.000000,0.464681
300,2.547800,2.590137,2.556355,1951831.000000,0.464922
350,2.499800,2.590247,2.517856,2274567.000000,0.465075
400,2.536900,2.587641,2.538273,2604322.000000,0.465487
450,2.515800,2.584139,2.539844,2933835.000000,0.465867
500,2.504900,2.582000,2.535208,3253334.000000,0.465851


('qwen-qlora-adapter/tokenizer_config.json',
 'qwen-qlora-adapter/special_tokens_map.json',
 'qwen-qlora-adapter/chat_template.jinja',
 'qwen-qlora-adapter/vocab.json',
 'qwen-qlora-adapter/merges.txt',
 'qwen-qlora-adapter/added_tokens.json',
 'qwen-qlora-adapter/tokenizer.json')

### Training Results Analysis

Based on the training metrics shown above, we can analyze several key aspects of the fine-tuning process:

#### Loss Progression Analysis

**Training Loss Trends:**
- **Initial**: 2.658 → **Final**: 2.449 (7.9% reduction)
- **Pattern**: Steady decline with some fluctuations, indicating healthy learning
- **Convergence**: Loss stabilizes around step 550-650, suggesting approaching optimal adaptation

**Validation Loss Behavior:**
- **Range**: 2.643 → 2.582 → 2.589 (slight increase at end)
- **Best Performance**: Step 500 (2.582) - model likely peaked here
- **Early Stopping Signal**: Validation loss plateauing/increasing after step 500 indicates potential overfitting

#### Key Performance Indicators

**Mean Token Accuracy:**
- **Progress**: 45.8% → 46.5% (0.7% improvement)
- **Interpretation**: Model correctly predicts ~46.5% of tokens, reasonable for summarization task
- **Context**: Token-level accuracy is challenging for generative tasks due to vocabulary diversity

**Entropy Analysis:**
- **Trend**: 2.654 → 2.489 (decreasing)
- **Meaning**: Model predictions becoming more confident and focused
- **Balance**: Lower entropy indicates better task specialization without over-confidence

#### Training Efficiency Metrics

**Tokens Processed:**
- **Total**: ~4.2M tokens across 650 steps
- **Rate**: ~6,500 tokens per step (consistent with batch size × sequence length)
- **Coverage**: Multiple passes through training data ensuring thorough learning

#### Quality Indicators

**Positive Signs:**
1. **Smooth Loss Decline**: No erratic jumps or instability
2. **Consistent Token Processing**: Stable throughput indicates no memory issues
3. **Validation Tracking**: Close monitoring prevents severe overfitting

**Areas of Concern:**
1. **Validation Loss Plateau**: After step 500, validation loss stops improving
2. **Potential Overfitting**: Training loss continues decreasing while validation stagnates
3. **Limited Accuracy Gains**: Token accuracy improvement is modest

#### Recommended Actions

**Immediate:**
- Use checkpoint from step 500 (best validation loss) rather than final checkpoint
- This demonstrates why `load_best_model_at_end=True` is crucial

**Future Experiments:**
- **Reduce Learning Rate**: Try 1e-4 instead of 2e-4 for more stable convergence
- **Early Stopping**: Set `early_stopping_patience=2` instead of 3 for quicker stopping
- **Regularization**: Increase LoRA dropout from 0.1 to 0.15 to reduce overfitting

**Advanced Analysis:**
- Plot learning curves to visualize training dynamics
- Compare against baseline model performance
- Analyze per-example loss distributions for data quality insights


## Model Deployment: Merging LoRA Adapters

### Deployment Strategy Options

After training, you have two deployment options:

1. **Adapter-based**: Keep base model and adapter separate (smaller storage, slower inference)
2. **Merged model**: Combine adapter weights into base model (larger storage, faster inference)

We'll demonstrate the merged approach for production deployment:

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

# Load the trained LoRA adapter with base model
print("Loading LoRA adapter...")
model = AutoPeftModelForCausalLM.from_pretrained(
    "qwen-qlora-adapter",               # Path to saved adapter
    low_cpu_mem_usage=True,             # Optimize memory during loading
    device_map="auto",                  # Distribute across available GPUs
)

# Merge LoRA weights into base model for deployment
print("Merging LoRA adapter with base model...")
merged_model = model.merge_and_unload()

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("qwen-qlora-adapter")

print("Model ready for inference!")
print(f"Merged model size: ~{sum(p.numel() for p in merged_model.parameters()):,} parameters")

## Model Inference and Evaluation

### Testing the Fine-Tuned Model

Let's evaluate our fine-tuned model's summarization capabilities with a controlled example:

In [ ]:
import torch

# Test article for summarization evaluation
article = """
The James Webb Space Telescope (JWST) has revolutionized our understanding of the cosmos since beginning operations in 2022. Recently, it captured an extraordinary image of the Pillars of Creation, a star-forming region located in the Eagle Nebula approximately 6,500 light-years from Earth. This iconic formation, previously photographed by the Hubble Space Telescope, appears dramatically different through Webb's infrared vision. The new images reveal previously hidden details of star formation processes, showing how dense clouds of gas and dust collapse under gravity to form new stellar systems. Webb's Near Infrared Camera (NIRCam) pierced through cosmic dust that typically obscures visible light observations, revealing thousands of newly formed stars that were invisible to previous telescopes. The telescope's unprecedented resolution and infrared capabilities allow astronomers to study the earliest stages of stellar evolution with remarkable clarity. These observations contribute significantly to our understanding of how stars and planetary systems form, providing insights that will inform astronomical research for decades to come.
"""

# Format input using the same instruction template as training
prompt = (
    "<|im_start|>user\n"
    "Summarize the following article:\n\n{article}<|im_end|>\n"
    "<|im_start|>assistant\n"
).format(article=article)

print("Input article length:", len(article.split()))
print("\nGenerating summary...")

# Tokenize input prompt
inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)

# Generate summary with controlled parameters
with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=100,                     # Limit summary length
        do_sample=True,                         # Enable sampling for variety
        temperature=0.7,                        # Balance creativity vs consistency
        top_p=0.9,                             # Nucleus sampling: focus on top 90% probability mass
        no_repeat_ngram_size=3,                # Prevent repetitive 3-gram phrases
        pad_token_id=tokenizer.eos_token_id    # Proper padding handling
    )

# Extract only the generated summary (excluding input prompt)
generated_text = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:], 
    skip_special_tokens=True
).strip()

print("Generated Summary:")
print(f'"{generated_text}"')
print(f"\nSummary length: {len(generated_text.split())} words")
print(f"Compression ratio: {len(article.split()) / len(generated_text.split()):.1f}x")

Generated Summary:
James Webb Space telescope has captured stunning new picture of the Milky Way .
It is called J-Warp and it was taken by NASA's space agency on Tuesday .


### Quantitative Evaluation on Test Set

For rigorous evaluation, we should assess model performance on our held-out test set using standard summarization metrics:


In [ ]:
from evaluate import load
import numpy as np
from tqdm import tqdm

# Load evaluation metrics
rouge = load("rouge")

def evaluate_model_on_test_set(model, tokenizer, test_dataset, num_samples=50):
    """
    Evaluate model performance on test set using ROUGE metrics.
    
    ROUGE metrics measure n-gram overlap between generated and reference summaries:
    - ROUGE-1: Unigram overlap (individual word matches)
    - ROUGE-2: Bigram overlap (two-word phrase matches)  
    - ROUGE-L: Longest common subsequence (captures sentence-level structure)
    """
    predictions = []
    references = []
    
    print(f"Evaluating on {num_samples} test examples...")
    
    for i in tqdm(range(min(num_samples, len(test_dataset)))):
        # Extract article from formatted text (before assistant response)
        full_text = test_dataset[i]["text"]
        article_start = full_text.find("Summarize the following article:\n\n") + len("Summarize the following article:\n\n")
        article_end = full_text.find("<|im_end|>", article_start)
        article = full_text[article_start:article_end].strip()
        
        # Extract reference summary (after assistant marker)
        ref_start = full_text.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
        ref_end = full_text.find("<|im_end|>", ref_start)
        reference = full_text[ref_start:ref_end].strip()
        
        # Generate summary
        prompt = (
            "<|im_start|>user\n"
            f"Summarize the following article:\n\n{article}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )
        
        prediction = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:], 
            skip_special_tokens=True
        ).strip()
        
        predictions.append(prediction)
        references.append(reference)
    
    # Calculate ROUGE scores
    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    
    return rouge_scores, predictions[:5], references[:5]  # Return sample outputs

# Run evaluation (uncomment to execute)
# rouge_scores, sample_preds, sample_refs = evaluate_model_on_test_set(merged_model, tokenizer, test_dataset)

# print("ROUGE Evaluation Results:")
# print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
# print(f"ROUGE-2: {rouge_scores['rouge2']:.4f}")  
# print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")

print("Evaluation function ready. Uncomment the lines above to run full evaluation.")


## Future Improvements and Advanced Techniques

### Hyperparameter Optimization

Our current configuration uses reasonable defaults, but systematic optimization could improve performance:

#### Learning Rate Scheduling
```python
# Current: Linear decay
# Alternatives to explore:
- Cosine annealing: lr_scheduler_type="cosine"
- Polynomial decay: lr_scheduler_type="polynomial"  
- Constant with warmup: lr_scheduler_type="constant_with_warmup"
```

#### LoRA Configuration Tuning
```python
# Grid search over key parameters:
lora_configs = [
    {"r": 16, "lora_alpha": 8},   # Lower rank, conservative
    {"r": 32, "lora_alpha": 16},  # Current configuration  
    {"r": 64, "lora_alpha": 32},  # Higher rank, more expressive
    {"r": 128, "lora_alpha": 64}, # Maximum expressiveness
]
```

#### Batch Size and Learning Rate Relationship
```python
# Scale learning rate with effective batch size:
effective_batch_size = per_device_batch_size * gradient_accumulation_steps * num_gpus
scaled_lr = base_lr * (effective_batch_size / 8)  # Linear scaling rule
```

### Advanced Training Techniques

#### 1. Gradient Accumulation Optimization
- **Current**: Fixed 4 steps
- **Improvement**: Dynamic accumulation based on GPU memory utilization
- **Benefit**: Maximize hardware utilization while preventing OOM

#### 2. Mixed Precision Training
- **Current**: FP16 throughout
- **Improvement**: BF16 for better numerical stability on modern hardware
- **Alternative**: Automatic mixed precision with loss scaling

#### 3. Advanced Optimizers
```python
# Current: paged_adamw_8bit
# Alternatives:
- "adafactor": Memory-efficient, good for large models
- "adamw_bnb_8bit": Standard 8-bit AdamW
- "lion": Recently proposed, potentially faster convergence
```

### Model Architecture Improvements

#### 1. Larger Base Models
- **Qwen2-1.5B-Instruct**: Better baseline capabilities
- **Qwen2-7B-Instruct**: Production-quality performance
- **Trade-off**: Higher memory requirements vs better quality

#### 2. Multi-LoRA Training
```python
# Train multiple adapters for different aspects:
task_configs = {
    "summarization": LoraConfig(target_modules=["q_proj", "v_proj"]),
    "style_adaptation": LoraConfig(target_modules=["gate_proj", "up_proj"]),
    "factual_accuracy": LoraConfig(target_modules=["k_proj", "o_proj"])
}
```

#### 3. QA-LoRA (Quantization-Aware LoRA)
- Train LoRA adapters with quantization-aware techniques
- Better performance when deploying quantized models
- Requires specialized training procedures

### Evaluation and Quality Improvements

#### 1. Comprehensive Evaluation Metrics
```python
evaluation_metrics = [
    "rouge",           # N-gram overlap
    "bleu",            # Machine translation metric adapted for summarization  
    "meteor",          # Semantic similarity with synonyms
    "bertscore",       # Contextual embedding similarity
    "factcc",          # Factual consistency checking
]
```

#### 2. Human Evaluation Framework
- **Fluency**: Is the summary grammatically correct and readable?
- **Coherence**: Does the summary flow logically?
- **Faithfulness**: Does it accurately represent the source?
- **Informativeness**: Does it capture key information?

#### 3. Automated Quality Checks
```python
# Implement automated quality filters:
quality_checks = {
    "length_ratio": (0.1, 0.3),      # Summary should be 10-30% of original
    "repetition_check": True,         # Flag excessive repetition
    "factual_consistency": True,      # Use NLI models for fact-checking
    "hallucination_detection": True,  # Detect fabricated information
}
```

### Production Deployment Considerations

#### 1. Model Serving Optimization
```python
# Optimize for inference speed:
deployment_config = {
    "torch_compile": True,           # PyTorch 2.0 compilation
    "flash_attention": True,         # Memory-efficient attention
    "quantization": "int8",          # Post-training quantization
    "tensor_parallelism": True,      # Multi-GPU inference
}
```

#### 2. A/B Testing Framework
- Compare fine-tuned model against baseline
- Measure user satisfaction and engagement
- Track business metrics (time saved, accuracy ratings)

#### 3. Continuous Learning Pipeline
- Collect user feedback on summaries
- Retrain models with human preferences (RLHF)
- Implement online learning for domain adaptation

### Research Extensions

#### 1. Domain-Specific Fine-Tuning
- **Medical**: PubMed abstracts → clinical summaries
- **Legal**: Court documents → case briefs  
- **Financial**: Earnings reports → investor summaries
- **Scientific**: Research papers → abstracts

#### 2. Multi-Modal Summarization
- Extend to include images, tables, charts
- Use vision-language models (LLaVA, GPT-4V)
- Handle complex document formats (PDFs, presentations)

#### 3. Controllable Summarization
```python
# Add control tokens for different summary styles:
control_prompts = {
    "length": "Generate a {short/medium/long} summary",
    "audience": "Summarize for {general/technical/executive} audience", 
    "focus": "Focus on {financial/technical/human-interest} aspects",
    "style": "Write in {formal/casual/bullet-point} style"
}
```